#### imports

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from pathlib import Path
from scipy.stats import gaussian_kde
import time
import math
import itertools
import pickle

import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import zuko


In [2]:
DATASET_PATH = Path("../../datasets/disasters_merged_all_feats.csv")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

In [3]:
TARGET_COLS = ["DAMAGES", "CASUALTIES"]
X_COLS = [
    'BEGIN_LAT', 'BEGIN_LON',
    'DURATION_HOURS', 'WIND_SPEED', # change with hail size for hail
    'TIME_DAY_SIN', 'TIME_DAY_COS', 'TIME_YEAR_NORM',
    'PRECIPITATION', 'TMIN', 'TMAX',
    'ELEVATION', 'SLOPE',
    'COV_BARREN', 'COV_CULTIVATED', 'COV_VEGETATION',
    'COV_FOREST', 'COV_WATER', 'COV_SNOW_ICE', 'COV_URBAN',
    'RIVER_DISTANCE', 'SEA_DISTANCE',
]
df_all = pd.read_csv(DATASET_PATH)

In [4]:
EPOCHS = 20
BATCH_SIZE = 512

In [ ]:
quantiles = [0.8,0.9,0.95,0.99]
lambdas = [0.0, 0.5, 1, 3]
event_groups = ['Drought', 'Flood', 'Freeze', 'Heat', 'Storm', 'Wildfire', 'Wind',]

### Functions

In [13]:
def get_data(event_group):
    data = {}
    df = df_all[df_all['EVENT_GROUP'] == event_group]
    X = df[X_COLS].values.astype(np.float32)
    Y = df[TARGET_COLS].values.astype(np.float32)
    Y = np.log1p(Y)
    X_train, X_temp, Y_train, Y_temp = train_test_split(X, Y, test_size=0.3, random_state=42)
    X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, test_size=0.5, random_state=42)

    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    X_train = x_scaler.fit_transform(X_train)
    X_test  = x_scaler.transform(X_test)
    X_val   = x_scaler.transform(X_val)
    Y_train = y_scaler.fit_transform(Y_train)
    Y_test  = y_scaler.transform(Y_test)
    Y_val   = y_scaler.transform(Y_val)
    X_train = torch.tensor(X_train).to(device)
    Y_train = torch.tensor(Y_train).to(device)
    X_val = torch.tensor(X_val).to(device)
    Y_val = torch.tensor(Y_val).to(device)
    X_test = torch.tensor(X_test).to(device)
    Y_test = torch.tensor(Y_test).to(device)
    train_ds   = TensorDataset(X_train, Y_train)
    val_ds   = TensorDataset(X_val, Y_val)
    test_ds   = TensorDataset(X_test, Y_test)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    test_loader   = DataLoader(test_ds, batch_size=BATCH_SIZE)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)
    data['train_loader'] = train_loader
    data['val_loader'] = val_loader
    data['test_loader'] = test_loader    
    data['x_scaler'] = x_scaler
    data['y_scaler'] = y_scaler
    
    return data

In [14]:
def compute_extreme_threshold(y, quantile=0.95):
    """
    Compute per-target threshold for extreme events automatically.
    y: [batch_size, num_targets]
    quantile: e.g., 0.95 for top 5% as extreme
    returns: threshold tensor [num_targets]
    """
    thresholds = torch.quantile(y, quantile, dim=0)
    return thresholds

BASE_LR = 1e-3

def train_model(data, params):

    flow = zuko.flows.NSF(
    features=2,
    context=21,
    transforms=8,
    hidden_features=[256, 256, 256],
    bins=16,
    )
    flow = flow.to(device)
    optimizer = torch.optim.Adam(flow.parameters(), lr=BASE_LR, weight_decay=1e-5)
    total_steps = EPOCHS * len(data["train_loader"])
    warmup_steps = 5 * len(data["train_loader"])

    best_val_loss = float("inf")
    best_state_dict = None
    global_step = 0


    for epoch in range(EPOCHS):
        flow.train()
        train_loss = 0.0

        for x, y in data["train_loader"]:
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)

            # Forward pass
            dist = flow(x)
            log_prob = dist.log_prob(y)  # [batch_size]

            # Compute per-target thresholds
            threshold_damage = compute_extreme_threshold(y[:, 0], quantile=params['quantiles'][0])
            threshold_casualty = compute_extreme_threshold(y[:, 1], quantile=params['quantiles'][1])

            # Identify extreme events per target
            extreme_damage = (y[:, 0] > threshold_damage).float()  # [batch_size]
            extreme_casualty = (y[:, 1] > threshold_casualty).float()  # [batch_size]

            # Combine weights per event
            weight = 1.0 + params['lambdas'][0] * extreme_damage + params['lambdas'][1] * extreme_casualty

            # Weighted negative log likelihood
            loss = -(weight * log_prob).mean()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            # Learning rate scheduling
            global_step += 1
            if global_step < warmup_steps:
                lr_scale = global_step / warmup_steps
            else:
                progress = (global_step - warmup_steps) / (total_steps - warmup_steps)
                lr_scale = 0.5 * (1 + math.cos(math.pi * progress))
            for pg in optimizer.param_groups:
                pg['lr'] = BASE_LR * lr_scale

        train_loss /= max(len(data["train_loader"]), 1)

        # Validation
        flow.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in data["val_loader"]:
                x = x.to(device)
                y = y.to(device)
                val_loss += -flow(x).log_prob(y).mean().item()
        val_loss /= max(len(data["val_loader"]), 1)

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = flow.state_dict()

    if best_state_dict is not None:
        flow.load_state_dict(best_state_dict)
    return flow

In [15]:
def compute_nll(flow, test_loader):
    flow.eval()
    nll = 0.0
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)
            nll += -flow(x).log_prob(y).mean().item()
    nll /= max(len(test_loader), 1)
    return nll

In [16]:
def compute_avg_true_pred_diff(flow, test_loader, y_scaler):
    flow.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            dist = flow(x_batch)
            y_hat = dist.sample().squeeze()
            y_true.append(y_batch.cpu())
            y_pred.append(y_hat.cpu())
    y_true = torch.cat(y_true, dim=0).numpy()
    y_pred = torch.cat(y_pred, dim=0).numpy()
    if y_true.ndim == 1:
        y_true = y_true.reshape(-1, 1)
    if y_pred.ndim == 1:
        y_pred = y_pred.reshape(-1, 1)
    y_true = y_scaler.inverse_transform(y_true)
    y_pred = y_scaler.inverse_transform(y_pred)
    y_pred = np.expm1(np.clip(y_pred, -20, None))
    y_true = np.expm1(np.clip(y_true, -20, None))
    avg_diff_damages = np.mean(np.abs(y_true[:, 0] - y_pred[:, 0]))
    avg_diff_casualties = np.mean(np.abs(y_true[:, 1] - y_pred[:, 1]))
    return avg_diff_damages, avg_diff_casualties

## Training - symmetric params

In [ ]:
pal_results = {}

for eg in event_groups:
    data = get_data(eg)
    for q, l in itertools.product(quantiles, lambdas):
        params = {
            "quantiles": [q, q],
            "lambdas": [l, l],
            "event_group": eg,
        }
        
        print(f"Training for {eg} with quantile={q} and lambda={l}...")

        flow = train_model(data, params)
        nll = compute_nll(flow, data["test_loader"])
        avg_diff_damages, avg_diff_casualties = compute_avg_true_pred_diff(flow, data["test_loader"], data["y_scaler"])
        
        pal_results[f"{eg}_{q}_{l}"] = {
            "params": params,
            "nll": nll,
            "avg_diff_damages": avg_diff_damages,
            "avg_diff_casualties": avg_diff_casualties,
        }

Training for Drought with quantile=0.8 and lambda=0.0...
Training for Drought with quantile=0.8 and lambda=0.5...
Training for Drought with quantile=0.8 and lambda=1...
Training for Drought with quantile=0.8 and lambda=3...
Training for Drought with quantile=0.9 and lambda=0.0...
Training for Drought with quantile=0.9 and lambda=0.5...
Training for Drought with quantile=0.9 and lambda=1...
Training for Drought with quantile=0.9 and lambda=3...
Training for Drought with quantile=0.95 and lambda=0.0...
Training for Drought with quantile=0.95 and lambda=0.5...
Training for Drought with quantile=0.95 and lambda=1...
Training for Drought with quantile=0.95 and lambda=3...
Training for Drought with quantile=0.99 and lambda=0.0...
Training for Drought with quantile=0.99 and lambda=0.5...
Training for Drought with quantile=0.99 and lambda=1...
Training for Drought with quantile=0.99 and lambda=3...
Training for Flood with quantile=0.8 and lambda=0.0...
Training for Flood with quantile=0.8 and 

In [ ]:
pickle.dump(pal_results, open("pal_results1.pkl", "wb"))

## Training - asymmetric params

In [19]:
pal_results = {}
event_groups = ['Wildfire',]
EPOCHS = 20

for eg in event_groups:
    data = get_data(eg)
    for q1,q2,l1,l2 in itertools.product(quantiles, quantiles, lambdas, lambdas):
        params = {
            "quantiles": [q1, q2],
            "lambdas": [l1, l2],
            "event_group": eg,
        }
        
        print(f"Training for {eg} with quantile={q1},{q2} and lambda={l1},{l2}...")

        flow = train_model(data, params)
        nll = compute_nll(flow, data["test_loader"])
        avg_diff_damages, avg_diff_casualties = compute_avg_true_pred_diff(flow, data["test_loader"], data["y_scaler"])
        
        pal_results[f"{eg}_{q1}_{q2}_{l1}_{l2}"] = {
            "params": params,
            "nll": nll,
            "avg_diff_damages": avg_diff_damages,
            "avg_diff_casualties": avg_diff_casualties,
        }

Training for Wildfire with quantile=0.8,0.8 and lambda=0.0,0.0...
Training for Wildfire with quantile=0.8,0.8 and lambda=0.0,0.5...
Training for Wildfire with quantile=0.8,0.8 and lambda=0.0,1...
Training for Wildfire with quantile=0.8,0.8 and lambda=0.0,3...
Training for Wildfire with quantile=0.8,0.8 and lambda=0.5,0.0...
Training for Wildfire with quantile=0.8,0.8 and lambda=0.5,0.5...
Training for Wildfire with quantile=0.8,0.8 and lambda=0.5,1...
Training for Wildfire with quantile=0.8,0.8 and lambda=0.5,3...
Training for Wildfire with quantile=0.8,0.8 and lambda=1,0.0...
Training for Wildfire with quantile=0.8,0.8 and lambda=1,0.5...
Training for Wildfire with quantile=0.8,0.8 and lambda=1,1...
Training for Wildfire with quantile=0.8,0.8 and lambda=1,3...
Training for Wildfire with quantile=0.8,0.8 and lambda=3,0.0...
Training for Wildfire with quantile=0.8,0.8 and lambda=3,0.5...
Training for Wildfire with quantile=0.8,0.8 and lambda=3,1...
Training for Wildfire with quantile=0.

In [20]:
pickle.dump(pal_results, open("pal_results2_w.pkl", "wb"))

## Results

In [ ]:
pal_results = pickle.load(open("pal_results1.pkl", "rb"))

In [41]:
# print top 3 best nll  
best_keys = sorted(pal_results, key=lambda k: pal_results[k]["nll"])[:3]
for i, key in enumerate(best_keys):
    print(f"Best {i+1} NLL: {pal_results[key]['nll']:.4f} for {key} with params: {pal_results[key]['params']}")

# print top 3 best avg_diff_damages
best_keys = sorted(pal_results, key=lambda k: pal_results[k]["avg_diff_damages"])[:3]
for i, key in enumerate(best_keys):
    print(f"Best {i+1} avg_diff_damages: {pal_results[key]['avg_diff_damages']:.4f} for {key} with params: {pal_results[key]['params']}")

# print top 3 best avg_diff_casualties
best_keys = sorted(pal_results, key=lambda k: pal_results[k]["avg_diff_casualties"])[:3]
for i, key in enumerate(best_keys):
    print(f"Best {i+1} avg_diff_casualties: {pal_results[key]['avg_diff_casualties']:.4f} for {key} with params: {pal_results[key]['params']}")


Best 1 NLL: -12.5383 for Storm_0.9_0.5 with params: {'quantiles': [0.9, 0.9], 'lambdas': [0.5, 0.5], 'event_group': 'Storm'}
Best 2 NLL: -12.1717 for Storm_0.8_3 with params: {'quantiles': [0.8, 0.8], 'lambdas': [3, 3], 'event_group': 'Storm'}
Best 3 NLL: -12.0846 for Storm_0.99_0.0 with params: {'quantiles': [0.99, 0.99], 'lambdas': [0.0, 0.0], 'event_group': 'Storm'}
Best 1 avg_diff_damages: 466.8542 for Heat_0.99_1 with params: {'quantiles': [0.99, 0.99], 'lambdas': [1, 1], 'event_group': 'Heat'}
Best 2 avg_diff_damages: 466.8550 for Heat_0.9_0.5 with params: {'quantiles': [0.9, 0.9], 'lambdas': [0.5, 0.5], 'event_group': 'Heat'}
Best 3 avg_diff_damages: 466.8550 for Heat_0.8_0.0 with params: {'quantiles': [0.8, 0.8], 'lambdas': [0.0, 0.0], 'event_group': 'Heat'}
Best 1 avg_diff_casualties: 0.0001 for Drought_0.95_3 with params: {'quantiles': [0.95, 0.95], 'lambdas': [3, 3], 'event_group': 'Drought'}
Best 2 avg_diff_casualties: 0.0002 for Drought_0.8_0.0 with params: {'quantiles': [

In [12]:
for eg in event_groups:
    eg_results = {k: v for k, v in pal_results.items() if k.startswith(eg)}
    print(f"\nResults for {eg}:")
    # best nll for each event group
    best_key = min(eg_results, key=lambda k: eg_results[k]["nll"])
    print(f"Best NLL: {eg_results[best_key]['nll']:.4f} with params: {eg_results[best_key]['params']}")
    # best avg_diff_damages for each event group
    best_key = min(eg_results, key=lambda k: eg_results[k]["avg_diff_damages"])
    print(f"Best avg_diff_damages: {eg_results[best_key]['avg_diff_damages']:.4f} with params: {eg_results[best_key]['params']}")
    # best avg_diff_casualties for each event group
    best_key = min(eg_results, key=lambda k: eg_results[k]["avg_diff_casualties"])
    print(f"Best avg_diff_casualties: {eg_results[best_key]['avg_diff_casualties']:.4f} with params: {eg_results[best_key]['params']}")



Results for Drought:
Best NLL: -11.5105 with params: {'quantiles': [0.95, 0.95], 'lambdas': [3, 3], 'event_group': 'Drought'}
Best avg_diff_damages: 63771.8867 with params: {'quantiles': [0.99, 0.99], 'lambdas': [0.5, 0.5], 'event_group': 'Drought'}
Best avg_diff_casualties: 0.0001 with params: {'quantiles': [0.95, 0.95], 'lambdas': [3, 3], 'event_group': 'Drought'}

Results for Flood:
Best NLL: -9.9473 with params: {'quantiles': [0.99, 0.99], 'lambdas': [3, 3], 'event_group': 'Flood'}
Best avg_diff_damages: 2594625.0000 with params: {'quantiles': [0.8, 0.8], 'lambdas': [0.0, 0.0], 'event_group': 'Flood'}
Best avg_diff_casualties: 0.0271 with params: {'quantiles': [0.8, 0.8], 'lambdas': [0.0, 0.0], 'event_group': 'Flood'}

Results for Freeze:
Best NLL: -12.0423 with params: {'quantiles': [0.95, 0.95], 'lambdas': [3, 3], 'event_group': 'Freeze'}
Best avg_diff_damages: 79511.1250 with params: {'quantiles': [0.9, 0.9], 'lambdas': [1, 1], 'event_group': 'Freeze'}
Best avg_diff_casualties:

In [ ]:
pal_results = pickle.load(open("pal_results2_w.pkl", "rb"))

In [21]:
for eg in event_groups:
    eg_results = {k: v for k, v in pal_results.items() if k.startswith(eg)}
    print(f"\nResults for {eg}:")
    # best nll for each event group
    best_key = min(eg_results, key=lambda k: eg_results[k]["nll"])
    print(f"Best NLL: {eg_results[best_key]['nll']:.4f} with params: {eg_results[best_key]['params']}")
    # best avg_diff_damages for each event group
    best_key = min(eg_results, key=lambda k: eg_results[k]["avg_diff_damages"])
    print(f"Best avg_diff_damages: {eg_results[best_key]['avg_diff_damages']:.4f} with params: {eg_results[best_key]['params']}")
    # best avg_diff_casualties for each event group
    best_key = min(eg_results, key=lambda k: eg_results[k]["avg_diff_casualties"])
    print(f"Best avg_diff_casualties: {eg_results[best_key]['avg_diff_casualties']:.4f} with params: {eg_results[best_key]['params']}")



Results for Wildfire:
Best NLL: -6.5625 with params: {'quantiles': [0.8, 0.95], 'lambdas': [1, 1], 'event_group': 'Wildfire'}
Best avg_diff_damages: 5983759.5000 with params: {'quantiles': [0.99, 0.99], 'lambdas': [0.0, 0.5], 'event_group': 'Wildfire'}
Best avg_diff_casualties: 0.3652 with params: {'quantiles': [0.8, 0.99], 'lambdas': [0.0, 0.0], 'event_group': 'Wildfire'}
